# Advanced 07 — World Models & Environment Modeling

**Scenario:** `deploy-1842` preceded Northstar Commerce's EU checkout degradation. We will compare rollback, disabling 3DS, shifting traffic, waiting, and a deliberately unsafe database rollback.

> **Safety boundary:** A world model is fallible decision support. Simulation can reject represented failures and compare alternatives; success does not prove production safety, grant capability, or authorize execution.

This credential-free notebook performs **zero production side effects**. Its deterministic fixtures use a fixed clock and seeded random generator.

You will:

1. separate sandbox, simulator, world model, and digital twin;
2. validate provenance, freshness, sensor quality, units, and model applicability;
3. keep observed and predicted state as different types;
4. compare point-estimate planning with uncertainty-aware Monte Carlo planning;
5. apply explicit utility, hard constraints, blast-radius, robustness, and sensitivity checks;
6. prove approval and fresh state are required after planning; and
7. evaluate calibration and controlled model promotion.

## Mental model and control boundaries

A **sandbox** isolates execution. A **simulator** predicts under assumptions. A **world model** represents selected state and transition dynamics. A **digital twin** synchronizes a scoped model with a real counterpart to a defined degree; it need not be deterministic or exact.

![World-model decision-support flow showing independent validity, approval, fresh-state, execution, and calibration boundaries.](assets/world-model-control-loop.svg)

The orange authority boundary is deliberately outside the simulator. `SIMULATION_PASS != APPROVED`.

## Reproducible setup

The reusable implementation lives in `policy.py` and `lab.py`. Pydantic models reject unknown fields, and the default path uses no provider SDK, network call, secret, or live service.

In [ ]:
import sys
from pathlib import Path

COURSE_DIR = Path("curriculum/advanced/07-world-models-environment-modeling").resolve()
if str(COURSE_DIR) not in sys.path:
    sys.path.insert(0, str(COURSE_DIR))

from policy import (
    ModelUpdateProposal,
    PlanningStatus,
    UtilityWeights,
    WorldModelPolicyError,
    assess_model_validity,
    calculate_calibration_metrics,
    detect_drift,
    observed_state_digest,
    promote_model,
    validate_model_update,
)
from lab import (
    FIXED_TIME,
    ACTION_PROFILE,
    action_proposals,
    approval_receipt,
    authorize_recommended_action,
    fixture_observed_outcome,
    model_snapshot,
    observed_state,
    planning_constraints,
    recommended_artifacts,
    run_planning_cycle,
    shadow_calibration_records,
)

print("Course directory:", COURSE_DIR.name)
print("Credential-free deterministic mode: ON")
print("Fixed clock:", FIXED_TIME.isoformat())

## Baseline: choose the largest point estimate

A weak baseline chooses the action with the highest fixture recovery probability. It ignores distributions, model applicability, affected tenants, data loss, worst cases, and authorization. We run it only to expose that limitation—never to execute anything.

In [ ]:
baseline = max(
    ACTION_PROFILE,
    key=lambda action: ACTION_PROFILE[action]["recovery_probability"],
)
print("Naive point-estimate winner:", baseline.value)
print("Recovery point estimate:", ACTION_PROFILE[baseline]["recovery_probability"])
print("Production side effects: 0")
assert baseline.value == "DATABASE_ROLLBACK"

The baseline selects database rollback because `0.90` is the largest single estimate. Later, hard data-loss and cross-tenant constraints will reject it. This is why a scalar score cannot carry the whole safety decision.

## Build observed state with provenance

`ObservedState` contains trusted application inputs. Every observation includes identity, source/version, observed/retrieved times, tenant, unit, value, and sensor quality. Simulator outputs use the separate `PredictedState` type.

In [ ]:
state = observed_state()
for item in state.observations:
    print(
        f"{item.variable:27} {item.value:8.1f} {item.unit:16} "
        f"quality={item.quality.value:5} source={item.source_version}"
    )
print("Observed-state digest:", observed_state_digest(state)[:16])

## Versioned snapshot and model-validity gate

A simulation must identify its model version, snapshot, calibration time, transition implementation, input digest, tenant, validated ranges, and seed. `assess_model_validity()` reports measurable ages plus an explicit status instead of hiding all uncertainty behind `confidence=0.72`.

In [ ]:
snapshot = model_snapshot(state)
validity = assess_model_validity(snapshot, state, now=FIXED_TIME)
print("Model version:", snapshot.model_version)
print("Snapshot:", snapshot.snapshot_id)
print("Validity:", validity.status.value)
print("Sensor / snapshot / model ages (s):", validity.oldest_sensor_age_seconds, validity.snapshot_age_seconds, validity.model_age_seconds)
assert validity.status.value == "VALID"

## Failure injection: OOD and missing telemetry

This model was validated up to 5,000 requests/second. A 7,000 request/second event is out of distribution. Missing critical telemetry makes model state incomplete. In both cases, the planning cycle stops before simulation.

In [ ]:
ood_state = observed_state(traffic_rps=7_000)
ood_decision = run_planning_cycle(ood_state, model_snapshot(ood_state))

from policy import SensorQuality
missing_state = observed_state(
    observation_overrides={"queue_depth": {"quality": SensorQuality.MISSING}}
)
missing_decision = run_planning_cycle(missing_state, model_snapshot(missing_state))

print("OOD:", ood_decision.status.value, ood_decision.reason_codes)
print("Missing sensor:", missing_decision.status.value, missing_decision.reason_codes)
assert ood_decision.simulation_results == ()
assert missing_decision.simulation_results == ()

## Typed proposals: planning permission is not execution authority

The planner may validly propose an approval-gated rollback. Parameters are action-specific; arbitrary SQL or shell strings are not authoritative production actions. Every proposal binds the model snapshot and observed-state digest.

In [ ]:
proposals = action_proposals(state, snapshot)
for proposal in proposals:
    print(
        f"{proposal.action_type.value:24} approval_required={str(proposal.approval_required):5} "
        f"capability={proposal.required_capability}"
    )
assert proposals[0].approval_required is True

## Seeded counterfactual scenario analysis

Counterfactual planning is not automatically Tree of Thoughts. Here it is deterministic fixture-driven Monte Carlo simulation. For each action, 200 samples vary dependency availability, capacity, latency, traffic, and recovery.

Utility uses explicit recovery, customer impact, data-loss risk, SLA exposure, reversibility, complexity, and uncertainty weights. Hard constraints independently block unacceptable outcomes.

In [ ]:
decision = run_planning_cycle(state, snapshot, samples=200)
print("Planning status:", decision.status.value)
print("Sensitivity winners:", decision.sensitivity_winners)
print()
print(f"{'action':24} {'P(recover)':>10} {'p50 min':>9} {'p90 min':>9} {'robust':>8} {'utility':>9}  constraints")
for result in decision.simulation_results:
    dist = result.distribution
    score = result.score
    constraints = ",".join(score.hard_constraint_violations) or "none"
    print(
        f"{score.action_type.value:24} {dist.recovery_probability:10.2f} "
        f"{dist.p50_recovery_minutes:9.1f} {dist.p90_recovery_minutes:9.1f} "
        f"{score.robustness_score:8.2f} {score.expected_utility:9.2f}  {constraints}"
    )

assert decision.status is PlanningStatus.READY_FOR_REVIEW

The database rollback has attractive recovery probability, but `DATA_LOSS_CONSTRAINT`, `BLAST_RADIUS_VIOLATION`, and `CROSS_TENANT_MUTATION` make it infeasible. Rollback deployment is the fixture's stable feasible winner, but the result says only that simulation supports a proposal.

## Sensitivity: a brittle winner should not become a recommendation

Tightening minimum robustness to 0.75 makes the normal scenario feasible but removes the winner under doubled traffic/latency. The planner returns `DECISION_UNSTABLE` and no recommendation.

In [ ]:
strict_constraints = planning_constraints().model_copy(
    update={"minimum_robustness": 0.75}
)
unstable = run_planning_cycle(
    state,
    snapshot,
    samples=200,
    sensitivity_multipliers=(1.0, 2.0),
    constraints=strict_constraints,
)
print("Status:", unstable.status.value)
print("Winners:", unstable.sensitivity_winners)
print("Recommendation:", unstable.recommended_proposal_id)
assert unstable.status is PlanningStatus.DECISION_UNSTABLE

## Approval and execution boundary

We first attempt to authorize the valid approval-gated plan without approval: it must fail. A validated receipt can create a typed execution envelope only when it binds the exact proposal, simulation, snapshot, state, tenant, action, target, policy, approver role, and time window—and the executor independently has the capability.

Creating the envelope is not a production side effect. Dispatch belongs to the hardened control plane from Course 03 and Advanced 05.

In [ ]:
proposal, simulation = recommended_artifacts(state, snapshot, decision)
try:
    authorize_recommended_action(state, decision, proposal, simulation, None)
except WorldModelPolicyError as error:
    print("Without approval:", error)
    assert str(error) == "APPROVAL_REQUIRED"

receipt = approval_receipt(proposal, simulation)
execution_envelope = authorize_recommended_action(
    state, decision, proposal, simulation, receipt
)
print("Validated envelope:", execution_envelope.execution_id)
print("Bound simulation:", execution_envelope.simulation_run_id)
print("Production side effects: 0")

## Synchronization race: approval becomes stale

The simulation used `deploy-1842`. If production is now `deploy-1843`, even an otherwise valid receipt cannot authorize the old proposal. The application returns `SIMULATION_STALE`; it must observe, snapshot, simulate, review, and approve again.

In [ ]:
changed_state = observed_state(deployment_version="deploy-1843")
try:
    authorize_recommended_action(
        changed_state, decision, proposal, simulation, receipt
    )
except WorldModelPolicyError as error:
    print("Changed production state:", error)
    assert str(error) == "SIMULATION_STALE"

## Prediction error, calibration, and drift

Relative error alone is not materiality. Compare absolute error, relative error, SLO impact, interval coverage, event calibration, and ranking sensitivity.

The next records are labelled **historical shadow-prediction fixtures**. The candidate model did not control production. A drift signal creates an update proposal; it does not silently modify the active model.

In [ ]:
records = shadow_calibration_records()
metrics = calculate_calibration_metrics(records)
drift = detect_drift(snapshot.model_version, metrics, now=FIXED_TIME)

print("Samples:", metrics.sample_count)
print("MAE / RMSE (min):", round(metrics.mae_minutes, 3), round(metrics.rmse_minutes, 3))
print("Interval coverage:", metrics.interval_coverage)
print("Brier score:", round(metrics.brier_score, 4))
print("Drift status:", drift.status.value)

## Controlled model update and backtest

`world-model-v13` must bind calibration lineage and pass historical prediction error, constraint-miss, ranking, and coverage gates. Promotion returns a new immutable snapshot; `world-model-v12` remains available for replay and rollback.

In [ ]:
update = ModelUpdateProposal(
    update_id="update-v13",
    current_model_version=snapshot.model_version,
    candidate_model_version="world-model-v13",
    calibration_record_ids=tuple(item.record_id for item in records),
    proposed_by="calibration-pipeline",
    created_at=FIXED_TIME,
)
report = validate_model_update(
    update,
    records,
    constraint_violation_miss_rate=0,
    ranking_accuracy=0.9,
    now=FIXED_TIME,
)
promoted = promote_model(
    snapshot,
    update,
    report,
    new_snapshot_id="snapshot-eu-1842-v13",
    new_state_digest="1" * 64,
    promoted_at=FIXED_TIME,
)
print("Backtest approved:", report.approved_for_promotion)
print("Old model:", snapshot.model_version)
print("New model:", promoted.model_version)
assert snapshot.model_version == "world-model-v12"

## Evaluation summary

| Question | Evidence in this notebook |
|---|---|
| Is the model applicable? | typed validity status, reason codes, and measured ages |
| What could happen? | seeded distributions with p10/p50/p90 and failure probability |
| Is an action feasible? | invariant, blast-radius, data-loss, worst-case, and robustness gates |
| Is the ranking stable? | sensitivity winners under changed assumptions |
| Is execution authorized now? | exact approval, capability, digest, and fresh-state binding |
| Does the model match reality? | MAE, RMSE, relative error, SLO impact, coverage, and Brier score |
| May a new model be trusted? | shadow evidence, historical backtest, validation report, versioned promotion |

Fixture metrics validate implementation behavior, not real incident performance or generalization.

## Production upgrade path

Replace fixed observations with authenticated tenant-scoped telemetry; fixed ranges with monitored applicability/OOD models; in-process simulation with isolated resource-bounded workers; hand-authored profiles with validated learned, causal, physics, or hybrid dynamics; and local receipts with an authoritative approval service.

Keep validity, constraints, capability, approval, fresh-state checks, dispatch, idempotency, reconciliation, audit, and model promotion application-owned. Never log secrets, sensitive raw payloads, or private model reasoning.

## Exercises

1. Add a typed capacity-scaling action with exact capability, parameters, and invariants.
2. Inject a 45-day-old dependency observation and reason about validity precedence if traffic is also OOD.
3. Add expected calibration error and compare it with Brier score.
4. Change utility weights and find a decision flip; decide whether to recommend or abstain.
5. Forge a receipt with the right text and wrong run ID; prove validation rejects it.

## Final principle

**A prediction is not an observation. A simulation is not an approval. A plan is not execution authority.**